# Template 07: SHAP Analysis

**Purpose:** Generate residual + SHAP range plots for top features

**Inputs:**
- results/06_shap_train.parquet
- results/06_shap_test.parquet
- results/05_predictions_train.parquet
- results/05_predictions_test.parquet
- data/04_train.parquet
- data/04_test.parquet

**Outputs:**
- plots/{dataset}_{feature}_residual.png
- plots/{dataset}_{feature}_shap_range.png
- results/07_analysis_summary.yaml

In [1]:
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_coll/v1"


In [3]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import yaml
import os
import sys
from pathlib import Path
from datetime import datetime

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment
from shap_utils import compute_shap_aggregate, create_residual_plot, create_shap_range_plot

print("########################################")
print("# STAGE 07: SHAP ANALYSIS")
print("########################################")

project_root = setup_notebook_environment()

########################################
# STAGE 07: SHAP ANALYSIS
########################################


In [4]:
with open(f"{config_path}/config.yaml") as f:
    cfg = yaml.safe_load(f)
output_base = cfg["paths"]["output_base"]
shap_cfg = cfg.get("shap_analysis", {})
top_n = shap_cfg.get("top_n_features", 40)
datasets = shap_cfg.get("datasets", ["train", "test"])
resid_cfg = shap_cfg.get("residual_plot", {})
shap_range_cfg = shap_cfg.get("shap_range_plot", {})
print(f"\nConfig: top_n={top_n}")


Config: top_n=40


In [5]:
plots_dir = f"{output_base}/plots"
os.makedirs(plots_dir, exist_ok=True)
print(f"Plots: {plots_dir}")

Plots: output/car_coll/v1/plots


In [6]:
print("\nLoading...")
shap_train = pd.read_parquet(f"{output_base}/results/06_shap_train.parquet")
shap_test = pd.read_parquet(f"{output_base}/results/06_shap_test.parquet")
preds_train = pd.read_parquet(f"{output_base}/results/05_predictions_train.parquet")
preds_test = pd.read_parquet(f"{output_base}/results/05_predictions_test.parquet")
df_train = pd.read_parquet(f"{output_base}/data/04_train.parquet")
df_test = pd.read_parquet(f"{output_base}/data/04_test.parquet")
print(f"SHAP: {shap_train.shape}, {shap_test.shape}")
print(f"Preds: {preds_train.shape}, {preds_test.shape}")


Loading...


SHAP: (10000, 91), (2707950, 91)
Preds: (2716120, 3), (2707950, 3)


In [7]:
# Merge train data with predictions
data_tr = df_train.loc[shap_train.index].copy()
data_tr['actual'] = preds_train.loc[shap_train.index, 'actual']
data_tr['pred'] = preds_train.loc[shap_train.index, 'pred']
data_tr['weight'] = preds_train.loc[shap_train.index, 'exposure']

fc_tr = shap_train.copy()
fc_tr['weight'] = data_tr['weight']
out_tr = data_tr
print(f"Train: {out_tr.shape}")

Train: (10000, 115)


In [8]:
# Merge test data with predictions
data_te = df_test.loc[shap_test.index].copy()
data_te['actual'] = preds_test.loc[shap_test.index, 'actual']
data_te['pred'] = preds_test.loc[shap_test.index, 'pred']
data_te['weight'] = preds_test.loc[shap_test.index, 'exposure']

fc_te = shap_test.copy()
fc_te['weight'] = data_te['weight']
out_te = data_te
print(f"Test: {out_te.shape}")

Test: (2707950, 115)


In [9]:
print("\nSHAP aggregates...")
shagg, shagg_num, shagg2 = compute_shap_aggregate(fc_tr, weight_col="weight")
print(f"Features: {len(shagg2)}")
print(shagg2.head(10))


SHAP aggregates...
Features: 70
                            field   total_shap
8           multi_pol_unknown_cal  1337.669831
87        cef_first_poten_dam_ind   615.289916
7                     married_ind   565.730029
89            cef_lien_holder_ind   458.582980
17                   vc_doors_raw   245.070576
74  vc_speed_sensitive_volume_raw   241.442687
66           vc_premium_audio_raw   209.124106
70              vc_rear_wiper_raw   204.486239
54     vc_heated_side_mirrors_raw   198.343443
79          vc_tinted_windows_raw   129.678268


In [10]:
features = shagg2["field"].head(top_n).tolist()
print(f"\nPlotting {len(features)} features")


Plotting 40 features


In [11]:
summary = {"timestamp": datetime.now().isoformat(), "plots": 0}

In [12]:
if "train" in datasets:
    print("="*60)
    print("TRAIN PLOTS")
    print("="*60)
    for i, f in enumerate(features, 1):
        try:
            print(f"[{i}/{len(features)}] {f}")
            if f not in out_tr.columns: continue
            fig, _ = create_residual_plot(out_tr, f, "weight", **resid_cfg)
            fig.savefig(f"{plots_dir}/train_{f}_residual.png", dpi=100, bbox_inches="tight")
            plt.close(fig)
            fig, _ = create_shap_range_plot(fc_tr, data_tr, f, "weight", **shap_range_cfg)
            fig.savefig(f"{plots_dir}/train_{f}_shap_range.png", dpi=100, bbox_inches="tight")
            plt.close(fig)
            summary["plots"] += 2
        except Exception as e:
            print(f"  ERROR: {e}")
    print("\n[OK] Train done")

TRAIN PLOTS
[1/40] multi_pol_unknown_cal
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[2/40] cef_first_poten_dam_ind
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[3/40] married_ind
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[4/40] cef_lien_holder_ind
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[5/40] vc_doors_raw
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[6/40] vc_speed_sensitive_volume_raw
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[7/40] vc_premium_audio_raw
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[8/40] vc_rear_wiper_raw
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[9/40] vc_heated_side_mirrors_raw
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[10/40] vc_tinted_windows_raw
  ERROR: "Label(s) ['den

In [13]:
if "test" in datasets:
    print("="*60)
    print("TEST PLOTS")
    print("="*60)
    for i, f in enumerate(features, 1):
        try:
            print(f"[{i}/{len(features)}] {f}")
            if f not in out_te.columns: continue
            fig, _ = create_residual_plot(out_te, f, "weight", **resid_cfg)
            fig.savefig(f"{plots_dir}/test_{f}_residual.png", dpi=100, bbox_inches="tight")
            plt.close(fig)
            fig, _ = create_shap_range_plot(fc_te, data_te, f, "weight", **shap_range_cfg)
            fig.savefig(f"{plots_dir}/test_{f}_shap_range.png", dpi=100, bbox_inches="tight")
            plt.close(fig)
            summary["plots"] += 2
        except Exception as e:
            print(f"  ERROR: {e}")
    print("\n[OK] Test done")

TEST PLOTS
[1/40] multi_pol_unknown_cal
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[2/40] cef_first_poten_dam_ind
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[3/40] married_ind
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[4/40] cef_lien_holder_ind
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[5/40] vc_doors_raw
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[6/40] vc_speed_sensitive_volume_raw
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[7/40] vc_premium_audio_raw
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[8/40] vc_rear_wiper_raw
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[9/40] vc_heated_side_mirrors_raw
  ERROR: "Label(s) ['denom', 'incurred_act', 'incurred_pred'] do not exist"
[10/40] vc_tinted_windows_raw
  ERROR: "Label(s) ['deno

In [14]:
summary["features"] = features
summary["top_n"] = top_n
summary_file = f"{output_base}/results/07_analysis_summary.yaml"
with open(summary_file, "w") as f:
    yaml.dump(summary, f)
print(f"\nSummary: {summary_file}")


Summary: output/car_coll/v1/results/07_analysis_summary.yaml


In [15]:
print("\n" + "#"*40)
print("# STAGE 07: COMPLETE")
print("#"*40)
print(f"Plots: {summary['plots']}")


########################################
# STAGE 07: COMPLETE
########################################
Plots: 0
